# SIG Datathon — Combined Pipeline
### Where should London build its next Tube station?

This notebook runs all five components of the analysis in sequence:

1. **Gravity Model** — Population demand scoring
2. **Connectivity Model** — Accessibility gap scoring  
3. **Crowding Reduction Model** — Congestion relief scoring
4. **Travel Time Reduction Model** — Journey-time improvement scoring
5. **Combined Scoring & Sensitivity Analysis** — Multi-criteria ranking

**Data source:** [github.com/ryan-lse/sig_datathon](https://github.com/ryan-lse/sig_datathon)


## 0. Setup & Environment Detection

In [ ]:
import os
import sys
import re
import math
import time
import copy
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from sklearn.neighbors import BallTree
from scipy.spatial import cKDTree

import folium
import branca.colormap as cm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

# ─── Environment detection ───
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    REPO_ROOT = "/kaggle/input/sig-datathon"  # adjust to your Kaggle dataset name
    WORK_DIR  = "/kaggle/working"
    print("Running on Kaggle")
else:
    # Local: assume notebook is at repo root or clone from GitHub
    if os.path.exists("Demand Prediction Model"):
        REPO_ROOT = "."
    elif os.path.exists("sig_datathon"):
        REPO_ROOT = "sig_datathon"
    else:
        print("Cloning repository from GitHub...")
        os.system("git clone https://github.com/ryan-lse/sig_datathon.git")
        REPO_ROOT = "sig_datathon"
    WORK_DIR = "."
    print(f"Running locally, REPO_ROOT = {os.path.abspath(REPO_ROOT)}")

# ─── Data paths ───
DEMAND_DATA   = os.path.join(REPO_ROOT, "Demand Prediction Model", "Data")
DEMAND_OUTPUT = os.path.join(REPO_ROOT, "Demand Prediction Model", "Output")
CONN_DATA     = os.path.join(REPO_ROOT, "Connectivity", "data")
CONN_MODEL    = os.path.join(REPO_ROOT, "Connectivity", "model")
CROWD_DATA    = os.path.join(REPO_ROOT, "Crowding_reduction", "data")
CROWD_OUTPUT  = os.path.join(REPO_ROOT, "Crowding_reduction", "output")
TRAVEL_DATA   = os.path.join(REPO_ROOT, "Travel  Time Reduction", "data")
TRAVEL_OUTPUT = os.path.join(REPO_ROOT, "Travel  Time Reduction", "output")
FINAL_OUTPUT  = os.path.join(REPO_ROOT, "final_weighting_score", "output")

os.makedirs(DEMAND_OUTPUT, exist_ok=True)
os.makedirs(CROWD_OUTPUT, exist_ok=True)
os.makedirs(TRAVEL_OUTPUT, exist_ok=True)
os.makedirs(FINAL_OUTPUT, exist_ok=True)

print("All paths configured.")


---
## 1. Gravity Model (Demand Prediction)

Identifies areas of London with the highest population density/influence — good candidates for a new station.

**How it works:**
- Loads population data by Output Area and geographic coordinates
- Creates a 1km × 1km grid across London
- Filters out grid points within 800m of existing stations
- For each grid point, computes a gravity score: sum of `population / distance^beta` for all people within a radius
- Log-transforms the score for normalisation


In [ ]:
# ─── 1.1 Load population and coordinate data ───
population = pd.read_csv(
    os.path.join(DEMAND_DATA, "Population.csv"), sep=";", usecols=[0, 2]
)
population = population.rename(columns={"All Ages": "Population", "OA11CD": "Area"})

coordinate = pd.read_csv(os.path.join(DEMAND_DATA, "Coordinate_filter.csv"), sep=",")

stations_demand = pd.read_csv(os.path.join(DEMAND_DATA, "combined_stations.csv"))

data = pd.merge(population, coordinate, on="Area")

print(f"Population records: {len(population)}")
print(f"Coordinate records: {len(coordinate)}")
print(f"Merged records:     {len(data)}")
print(f"Existing stations:  {len(stations_demand)}")


In [ ]:
# ─── 1.2 Build spatial data ───
geometry = [Point(xy) for xy in zip(data["LONG"], data["LAT"])]
gdf = gpd.GeoDataFrame(data, geometry=geometry, crs="EPSG:4326")

station_geometry = [Point(xy) for xy in zip(stations_demand["lon"], stations_demand["lat"])]
stations_gdf = gpd.GeoDataFrame(stations_demand, geometry=station_geometry, crs="EPSG:4326")

# Convert to British National Grid (meters)
gdf = gdf.to_crs(epsg=27700)
stations_gdf = stations_gdf.to_crs(epsg=27700)

print("CRS converted to EPSG:27700 (British National Grid)")


In [ ]:
# ─── 1.3 Generate candidate grid ───
xmin, ymin, xmax, ymax = gdf.total_bounds
step = 1000  # 1 km grid
grid_points = [Point(x, y) for x in np.arange(xmin, xmax, step)
                            for y in np.arange(ymin, ymax, step)]
grid = gpd.GeoDataFrame(geometry=grid_points, crs=gdf.crs)

print(f"Initial grid points: {len(grid)}")

# Filter out grid points within 800m of existing stations
coords = np.array([[pt.x, pt.y] for pt in gdf.geometry])
pop = gdf["Population"].values
grid_coords = np.array([[pt.x, pt.y] for pt in grid.geometry])
station_coords = np.array([[pt.x, pt.y] for pt in stations_gdf.geometry])

station_tree = BallTree(station_coords, metric="euclidean")
idxs = station_tree.query_radius(grid_coords, r=800)
mask = np.array([len(i) == 0 for i in idxs])
grid = grid[mask].reset_index(drop=True)
grid_coords = np.array([[pt.x, pt.y] for pt in grid.geometry])

print(f"Grid points after 800m station filter: {len(grid)}")


In [ ]:
# ─── 1.4 Compute gravity scores ───
tree = BallTree(coords, leaf_size=50, metric="euclidean")

radius = 2500
beta = 1.5

idxs_list, dists_list = tree.query_radius(grid_coords, r=radius, return_distance=True)

gravity_scores = []
for idxs, d in zip(idxs_list, dists_list):
    idxs = np.array(idxs, dtype=int)
    d = np.array(d)
    if len(idxs) == 0:
        gravity_scores.append(0)
        continue
    d = np.where(d == 0, 1e-6, d)
    score = (pop[idxs] / (d ** beta)).sum()
    gravity_scores.append(score)

gravity_scores = np.array(gravity_scores)
grid["gravity_score"] = gravity_scores
grid["gravity_log"] = np.log1p(grid["gravity_score"])

# Convert to lat/lon
grid_latlon = grid.to_crs(epsg=4326)
grid_latlon["lat"] = grid_latlon.geometry.y
grid_latlon["lon"] = grid_latlon.geometry.x
grid_latlon = grid_latlon.sort_values("gravity_log", ascending=False).reset_index(drop=True)

print(f"Gravity scores computed for {len(grid_latlon)} grid points")
print(f"gravity_log range: {grid_latlon['gravity_log'].min():.4f} – {grid_latlon['gravity_log'].max():.4f}")

# Save
sorted_grid_path = os.path.join(DEMAND_OUTPUT, "sorted_grid.csv")
grid_latlon[["lat", "lon", "gravity_score", "gravity_log"]].to_csv(sorted_grid_path, index=False)
print(f"Saved: {sorted_grid_path}")

# Top 10
top10_gravity = grid_latlon.head(10)
print("\nTop 10 by gravity score:")
print(top10_gravity[["lat", "lon", "gravity_log"]].to_string(index=False))


### 1.5 Gravity Score Heatmap

In [ ]:
# ─── Gravity city-wide heatmap ───
center_lat = float(grid_latlon["lat"].mean())
center_lon = float(grid_latlon["lon"].mean())

m_gravity = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_gravity)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_gravity)

vmin = float(grid_latlon["gravity_log"].min())
vmax = float(grid_latlon["gravity_log"].max())

cmap_gravity = cm.LinearColormap(
    ["#08306b", "#2171b5", "#6baed6", "#ef6548", "#cb181d", "#99000d"],
    vmin=vmin, vmax=vmax,
)
cmap_gravity.caption = "Gravity Score (log)"

for _, r in grid_latlon.iterrows():
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=5.0,
        color=cmap_gravity(float(r["gravity_log"])),
        fill=True, fill_opacity=0.8, weight=0,
        popup=(
            f"gravity_score={r['gravity_score']:.2f}<br>"
            f"gravity_log={r['gravity_log']:.4f}<br>"
            f"lat={r['lat']:.6f}, lon={r['lon']:.6f}"
        ),
    ).add_to(m_gravity)

cmap_gravity.add_to(m_gravity)
folium.LayerControl(collapsed=False).add_to(m_gravity)

print("Gravity city-wide heatmap:")
m_gravity


### 1.6 Gravity Top 10 Map

In [ ]:
# ─── Gravity top 10 map ───
top10 = grid_latlon.head(10).reset_index(drop=True)

m_top = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_top)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_top)

for _, r in grid_latlon.iterrows():
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=4.0,
        color=cmap_gravity(float(r["gravity_log"])),
        fill=True, fill_opacity=0.25, weight=0,
    ).add_to(m_top)

for rank, (_, row) in enumerate(top10.iterrows(), start=1):
    lat, lon = float(row["lat"]), float(row["lon"])
    folium.CircleMarker(
        location=[lat, lon], radius=16,
        color="#d7191c", fill=True, fill_color="#d7191c",
        fill_opacity=0.85, weight=2,
        popup=folium.Popup(
            f"<b>Rank #{rank}</b><br>"
            f"gravity_log={row['gravity_log']:.4f}<br>"
            f"lat={row['lat']:.6f}, lon={row['lon']:.6f}",
            max_width=300,
        ),
    ).add_to(m_top)
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:12px;font-weight:bold;color:white;text-align:center;line-height:26px;width:26px;height:26px;">{rank}</div>',
            icon_size=(26, 26), icon_anchor=(13, 13),
        ),
    ).add_to(m_top)

folium.LayerControl(collapsed=False).add_to(m_top)
print("Gravity top 10 map:")
m_top


---
## 2. Connectivity Model

Measures how poorly served each grid point is by the existing rail network.

**How it works:**
- Loads Underground, Elizabeth line, and DLR station locations
- For each grid point, finds the k=5 nearest stations
- Computes harmonic mean of adjusted distances (higher = further from stations = worse served)
- Computes line diversity (fewer unique lines nearby = worse served)
- Combines both into a connectivity score (higher = bigger gap = better candidate)


In [ ]:
# ─── 2.1 Connectivity helper functions ───
# (from Connectivity/model/dist_func.py)

def nearest_neighbor(centroids, stations, k=3, correction_factor=1.3):
    if isinstance(centroids, gpd.GeoDataFrame) and "geometry" in centroids.columns:
        centroid_coords = np.column_stack((centroids.geometry.x, centroids.geometry.y))
        result = pd.DataFrame({"lat": centroids["lat"], "lon": centroids["lon"]})
    else:
        print("input centroids is not in geopandas format")
        return None

    if isinstance(stations, gpd.GeoDataFrame) and "geometry" in stations.columns:
        station_coords = np.column_stack((stations.geometry.x, stations.geometry.y))
    else:
        print("input stations is not in geopandas format")
        return None

    print(f"Total {len(centroid_coords)} grid points")
    print(f"Mapping {len(stations)} stations")

    tree = cKDTree(station_coords)
    dist, idx = tree.query(centroid_coords, k=k)

    for i in range(k):
        result[f"nearest_station_{i+1}"] = stations.iloc[idx[:, i]]["NAME"].values
        result[f"nearest_station_{i+1}_dist"] = dist[:, i] * correction_factor
        result[f"nearest_station_{i+1}_lat"] = stations.iloc[idx[:, i]]["lat"].values
        result[f"nearest_station_{i+1}_long"] = stations.iloc[idx[:, i]]["lon"].values

    if k >= 1:
        dist_cols = [f"nearest_station_{i+1}_dist" for i in range(k)]
        with np.errstate(divide="ignore", invalid="ignore"):
            reciprocal_sum = (1 / result[dist_cols]).sum(axis=1)
            result["harmonic_mean_adj_dist"] = k / reciprocal_sum

        hm = result["harmonic_mean_adj_dist"]
        hm_min, hm_max = hm.min(), hm.max()
        if hm_max > hm_min:
            result["harmonic_mean_adj_dist_norm"] = (hm - hm_min) / (hm_max - hm_min)
        else:
            result["harmonic_mean_adj_dist_norm"] = 0.0

    return result


def find_line_diversity(nearest_df, stations, k=3, station_col="NAME", lines_col="LINES"):
    station_cols = [f"nearest_station_{i+1}" for i in range(k)]

    def _parse_lines(value):
        if isinstance(value, list):
            return [str(x).strip() for x in value if str(x).strip()]
        if pd.isna(value):
            return []
        return [x.strip() for x in str(value).split(",") if x.strip()]

    station_to_lines = {
        row[station_col]: _parse_lines(row[lines_col])
        for _, row in stations[[station_col, lines_col]].drop_duplicates(subset=[station_col]).iterrows()
        if lines_col in stations.columns
    }

    result = nearest_df.copy()
    unique_counts = []
    for _, row in result[station_cols].iterrows():
        pooled = []
        for name in row.values:
            pooled.extend(station_to_lines.get(name, []))
        unique_counts.append(len(set(pooled)))

    result["line_unique_count"] = unique_counts
    lc = result["line_unique_count"]
    lc_min, lc_max = lc.min(), lc.max()
    if lc_max > lc_min:
        result["line_unique_count_norm"] = 1 - ((lc - lc_min) / (lc_max - lc_min))
    else:
        result["line_unique_count_norm"] = 0.0

    return result


def combine_connectivity_score(result_df, alpha=0.5, beta=0.5):
    result = result_df.copy()
    result["connectivity_score"] = (
        alpha * result["harmonic_mean_adj_dist_norm"]
        + beta * result["line_unique_count_norm"]
    )
    return result

print("Connectivity functions defined.")


In [ ]:
# ─── 2.2 Load station data ───
gdf_underground = gpd.read_file(os.path.join(CONN_DATA, "underground.geojson"))
gdf_elizabeth = gpd.read_file(os.path.join(CONN_DATA, "Elizabeth.geojson"))
gdf_dlr = gpd.read_file(os.path.join(CONN_DATA, "DLR.geojson"))
grids = gpd.read_file(os.path.join(CONN_DATA, "grids.geojson"))

# Fix CRS: source files contain EPSG:27700 coordinates
gdf_underground = gdf_underground.set_crs(epsg=27700, allow_override=True)
gdf_elizabeth = gdf_elizabeth.set_crs(epsg=27700, allow_override=True)
gdf_dlr = gdf_dlr.set_crs(epsg=27700, allow_override=True)

# Remove duplicate stations
gdf_underground = gdf_underground.drop_duplicates(subset=["NAME"], keep="first").reset_index(drop=True)

def add_lat_lon_columns(gdf_in):
    gdf_out = gdf_in.copy()
    gdf_wgs84 = gdf_out.to_crs(epsg=4326)
    gdf_out["lat"] = gdf_wgs84.geometry.y
    gdf_out["lon"] = gdf_wgs84.geometry.x
    return gdf_out

gdf_underground = add_lat_lon_columns(gdf_underground)
gdf_elizabeth = add_lat_lon_columns(gdf_elizabeth)
gdf_dlr = add_lat_lon_columns(gdf_dlr)

# Add LINES column to DLR (missing from source) so line diversity works
if "LINES" not in gdf_dlr.columns:
    gdf_dlr["LINES"] = "DLR"

# Combined station dataset
gdf_stations = pd.concat([gdf_underground, gdf_elizabeth, gdf_dlr], ignore_index=True)
gdf_stations = gpd.GeoDataFrame(gdf_stations, geometry="geometry", crs=gdf_underground.crs)

print(f"Underground: {len(gdf_underground)}")
print(f"Elizabeth:   {len(gdf_elizabeth)}")
print(f"DLR:         {len(gdf_dlr)}")
print(f"Combined:    {len(gdf_stations)}")


In [ ]:
# ─── 2.3 Compute connectivity scores ───
conn_result = nearest_neighbor(grids, gdf_stations, k=5)
conn_result = find_line_diversity(conn_result, gdf_stations, k=5)
conn_result = combine_connectivity_score(conn_result, alpha=0.5, beta=0.5)

print(f"connectivity_score range: {conn_result['connectivity_score'].min():.4f} to {conn_result['connectivity_score'].max():.4f}")

# Save
conn_csv_path = os.path.join(REPO_ROOT, "Connectivity", "connectivity_scores.csv")
conn_result.to_csv(conn_csv_path, index=False)
print(f"Saved: {conn_csv_path}")

conn_result.head()


### 2.4 Connectivity Score Heatmap

In [ ]:
# ─── Connectivity city-wide heatmap ───
if isinstance(conn_result, gpd.GeoDataFrame) and "geometry" in conn_result.columns and conn_result.crs is not None:
    result_map = conn_result.to_crs(epsg=4326).copy()
elif {"lon", "lat"}.issubset(conn_result.columns):
    is_projected = (conn_result["lon"].abs().max() > 180) or (conn_result["lat"].abs().max() > 90)
    in_crs = "EPSG:27700" if is_projected else "EPSG:4326"
    result_map = gpd.GeoDataFrame(
        conn_result.copy(),
        geometry=gpd.points_from_xy(conn_result["lon"], conn_result["lat"]),
        crs=in_crs,
    ).to_crs(epsg=4326)

center_lat_c = float(result_map.geometry.y.mean())
center_lon_c = float(result_map.geometry.x.mean())

m_conn = folium.Map(location=[center_lat_c, center_lon_c], zoom_start=10, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_conn)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_conn)

vmin_c = float(result_map["connectivity_score"].min())
vmax_c = float(result_map["connectivity_score"].max())
cmap_conn = cm.LinearColormap(
    ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"],
    vmin=vmin_c, vmax=vmax_c,
)
cmap_conn.caption = "Connectivity Score"

for _, r in result_map.iterrows():
    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=5.0,
        color=cmap_conn(float(r["connectivity_score"])),
        fill=True, fill_opacity=0.8, weight=0,
        popup=(
            f"score={r['connectivity_score']:.3f}<br>"
            f"line_unique={r.get('line_unique_count', '')}<br>"
            f"nearest={r.get('nearest_station_1', 'NA')}"
        ),
    ).add_to(m_conn)

cmap_conn.add_to(m_conn)
folium.LayerControl(collapsed=False).add_to(m_conn)

print("Connectivity city-wide heatmap:")
m_conn


---
## 3. Crowding Reduction Model

Estimates how much passenger congestion each candidate location could relieve from nearby overloaded stations.

**How it works:**
- Loads station locations and annual passenger flow data
- For each candidate, finds stations within 2km
- Computes diverted flow: `alpha * flow * exp(-beta * distance)`
- Higher score = more passengers diverted = better candidate for congestion relief


In [ ]:
# ─── 3.1 Data preparation: clean station names ───

def clean_station_name(x):
    x = str(x).lower().strip()
    x = re.sub(r"[''']", "", x)
    x = re.sub(r"\.", "", x)
    x = re.sub(r"\s*-\s*dlr$", "", x)
    x = re.sub(r"\s+(lu|lo|dlr|nr|el|tfl|ezl)$", "", x)
    x = x.replace("bank and monument", "bank")
    x = re.sub(r"\s+", " ", x)
    return x


def pretty_station_name(name):
    name = str(name).strip().title()
    fixes = {
        "King'S Cross": "King's Cross",
        "St Johns Wood": "St John's Wood",
        "Earls Court": "Earl's Court",
        "Bromley-By-Bow": "Bromley-by-Bow",
    }
    for bad, good in fixes.items():
        name = name.replace(bad, good)
    return name


# Load station CSVs
underground_c = pd.read_csv(os.path.join(CROWD_DATA, "Underground_Stations.csv"))
overground_c = pd.read_csv(os.path.join(CROWD_DATA, "Overground_Stations.csv"))
dlr_c = pd.read_csv(os.path.join(CROWD_DATA, "DLR_Stations.csv"))
elizabeth_c = pd.read_csv(os.path.join(CROWD_DATA, "Elizabeth_Line_Stations.csv"))

# Convert each to GeoDataFrame in EPSG:27700, then to WGS84
for df_name, df_val in [("underground", underground_c), ("overground", overground_c),
                         ("dlr", dlr_c), ("elizabeth", elizabeth_c)]:
    gdf_tmp = gpd.GeoDataFrame(
        df_val, geometry=gpd.points_from_xy(df_val["X"], df_val["Y"]), crs="EPSG:27700"
    ).to_crs("EPSG:4326")
    df_val["lon"] = gdf_tmp.geometry.x
    df_val["lat"] = gdf_tmp.geometry.y

underground_c["mode"] = "underground"
overground_c["mode"] = "overground"
dlr_c["mode"] = "dlr"
elizabeth_c["mode"] = "elizabeth"

stations_crowd = gpd.GeoDataFrame(
    pd.concat([underground_c, overground_c, dlr_c, elizabeth_c], ignore_index=True),
    geometry=gpd.points_from_xy(
        pd.concat([underground_c, overground_c, dlr_c, elizabeth_c], ignore_index=True)["lon"],
        pd.concat([underground_c, overground_c, dlr_c, elizabeth_c], ignore_index=True)["lat"],
    ),
    crs="EPSG:4326",
)

print(f"Combined station rows: {len(stations_crowd)}")


In [ ]:
# ─── 3.2 Load and merge passenger flows ───
flows = pd.read_excel(os.path.join(CROWD_DATA, "AC2024_AnnualisedEntryExit_Public.xlsx"), header=5)

stations_crowd["station"] = stations_crowd["NAME"].apply(clean_station_name)
flows["station"] = flows["Station"].apply(clean_station_name)
flows = flows[["station", "Annualised"]].copy()
flows["Annualised"] = pd.to_numeric(flows["Annualised"], errors="coerce")
flows = flows.groupby("station", as_index=False)["Annualised"].sum()

stations_crowd = stations_crowd.merge(flows, on="station", how="left")

# Collapse duplicate station rows
stations_crowd = stations_crowd.groupby("station").agg({
    "lon": "first", "lat": "first", "Annualised": "first", "geometry": "first"
}).reset_index()
stations_crowd = gpd.GeoDataFrame(stations_crowd, geometry="geometry", crs="EPSG:4326")

# Keep only stations with valid flow
stations_crowd = stations_crowd.dropna(subset=["Annualised"]).copy()
stations_crowd["Annualised"] = pd.to_numeric(stations_crowd["Annualised"], errors="coerce")
stations_crowd = stations_crowd.dropna(subset=["Annualised"]).copy()
stations_crowd["station"] = stations_crowd["station"].apply(pretty_station_name)

print(f"Stations with valid flow data: {len(stations_crowd)}")
print(f"Total annualised passengers: {stations_crowd['Annualised'].sum():,.0f}")


In [ ]:
# ─── 3.3 Haversine distance function ───

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

def find_nearest_station(lat, lon, station_lats, station_lons, station_names):
    distances = haversine(lat, lon, station_lats, station_lons)
    idx = np.argmin(distances)
    return station_names[idx], distances[idx]

print("Distance functions defined.")


In [ ]:
# ─── 3.4 Compute crowding scores ───

# Use the same grid as gravity model
candidates_crowd = grid_latlon[["lat", "lon"]].copy()
candidates_crowd = gpd.GeoDataFrame(
    candidates_crowd,
    geometry=gpd.points_from_xy(candidates_crowd["lon"], candidates_crowd["lat"]),
    crs="EPSG:4326",
)

# Model parameters
alpha_c = 0.25
beta_c = 1.5
radius_km = 2.0
min_station_distance_km = 0.8

station_lats = stations_crowd["lat"].to_numpy()
station_lons = stations_crowd["lon"].to_numpy()
station_flows = stations_crowd["Annualised"].to_numpy()
station_names = stations_crowd["station"].to_numpy()

scores, top_contributors, min_distances, valid_candidate = [], [], [], []

for idx, candidate in candidates_crowd.iterrows():
    cand_lat, cand_lon = candidate["lat"], candidate["lon"]
    distances = haversine(cand_lat, cand_lon, station_lats, station_lons)
    min_d = distances.min()
    min_distances.append(min_d)

    if min_d < min_station_distance_km:
        scores.append(0.0)
        top_contributors.append("")
        valid_candidate.append(False)
    else:
        local_mask = distances < radius_km
        if local_mask.sum() == 0:
            scores.append(0.0)
            top_contributors.append("")
            valid_candidate.append(True)
        else:
            local_d = distances[local_mask]
            local_f = station_flows[local_mask]
            local_n = station_names[local_mask]
            diverted = alpha_c * local_f * np.exp(-beta_c * local_d)
            scores.append(diverted.sum())
            valid_candidate.append(True)
            contrib = pd.DataFrame({"station": local_n, "diverted": diverted})
            contrib = contrib.sort_values("diverted", ascending=False)
            top_contributors.append(", ".join(contrib["station"].head(3).tolist()))

    if idx % 500 == 0:
        print(f"Processed {idx} / {len(candidates_crowd)} candidates")

candidates_crowd["crowding_score"] = scores
candidates_crowd["top_relieved_stations"] = top_contributors
candidates_crowd["min_distance_to_station_km"] = min_distances
candidates_crowd["valid_candidate"] = valid_candidate

# Normalise
s_min, s_max = candidates_crowd["crowding_score"].min(), candidates_crowd["crowding_score"].max()
candidates_crowd["crowding_score_norm"] = (candidates_crowd["crowding_score"] - s_min) / (s_max - s_min)

# Save
crowd_path = os.path.join(CROWD_OUTPUT, "team_crowding_scores.csv")
candidates_crowd[["lat", "lon", "crowding_score", "crowding_score_norm",
                   "min_distance_to_station_km", "top_relieved_stations"]].to_csv(crowd_path, index=False)
print(f"\nSaved: {crowd_path}")

top10_crowd = candidates_crowd.sort_values("crowding_score", ascending=False).head(10)
print("\nTop 10 by crowding relief:")
print(top10_crowd[["lat", "lon", "crowding_score", "top_relieved_stations"]].to_string(index=False))


### 3.5 Crowding Score Map

In [ ]:
# ─── Crowding top 20 map ───
top20_crowd = candidates_crowd.sort_values("crowding_score", ascending=False).head(20)

m_crowd = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_crowd)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_crowd)

# Background: all stations
for _, s in stations_crowd.iterrows():
    folium.CircleMarker(
        location=[float(s["lat"]), float(s["lon"])],
        radius=4, color="gray", fill=True, fill_opacity=0.4, weight=0,
    ).add_to(m_crowd)

# Top 20 candidates
for rank, (_, row) in enumerate(top20_crowd.iterrows(), start=1):
    folium.CircleMarker(
        location=[float(row["lat"]), float(row["lon"])],
        radius=12, color="red", fill=True, fill_color="red",
        fill_opacity=0.85, weight=2,
        popup=folium.Popup(
            f"<b>#{rank}</b><br>crowding={row['crowding_score']:,.0f}<br>{row['top_relieved_stations']}",
            max_width=300,
        ),
    ).add_to(m_crowd)
    folium.Marker(
        location=[float(row["lat"]), float(row["lon"])],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;font-weight:bold;color:white;text-align:center;line-height:22px;width:22px;height:22px;">{rank}</div>',
            icon_size=(22, 22), icon_anchor=(11, 11),
        ),
    ).add_to(m_crowd)

folium.LayerControl(collapsed=False).add_to(m_crowd)
print("Crowding top 20 map:")
m_crowd


---
## 4. Travel Time Reduction Model

Estimates total person-seconds of travel time saved by adding a candidate station to the network.

**How it works:**
- Builds a graph of the London rail network from link frequency data
- For each candidate, connects it to its 3 nearest stations
- Computes shortest paths before and after adding the station
- Weights time savings by passenger flow at origin stations

**Note:** This section uses the TfL Journey API and takes ~1 hour for 108 candidates. 
It will use a cached results file if available.


In [ ]:
# ─── 4.1 Configuration ───
import networkx as nx

tfl_api_key = "a4866fe16555474daef639d13346f389"

# Link frequency data
link_freq_file = "numbat2024MONlinkfreqdata(UsingThisOne).csv"
link_freq_path = os.path.join(TRAVEL_DATA, link_freq_file)

# Candidate stations: top 100 gravity + 8 extras
top_n_travel = 100
grid_travel = grid_latlon.head(top_n_travel)

candidate_stations = [
    (row.lat, row.lon, f"Grid_{i}")
    for i, row in grid_travel.iterrows()
]

extra_candidates = [
    (51.4999, -0.0384, "Extra_Deptford"),
    (51.4802,  0.0615, "Extra_Woolwich"),
    (51.4280, -0.0415, "Extra_Sydenham"),
    (51.3838, -0.0865, "Extra_WestCroydon1"),
    (51.4455, -0.0120, "Extra_Catford"),
    (51.3930, -0.1005, "Extra_WestCroydon2"),
    (51.3660, -0.1016, "Extra_WestCroydon3"),
    (51.3748, -0.0869, "Extra_WestCroydon4"),
]

# Check which extras are already in top 100
existing_coords = set((round(lat, 3), round(lon, 3)) for lat, lon, _ in candidate_stations)
extras_to_add = [(lat, lon, lbl) for lat, lon, lbl in extra_candidates
                  if (round(lat, 3), round(lon, 3)) not in existing_coords]
candidate_stations.extend(extras_to_add)

print(f"Total candidates: {len(candidate_stations)} (top {top_n_travel} gravity + {len(extras_to_add)} extra)")

n_connections = 3
benefit_radius_km = 2.5
min_nearby_stations = 3

cache_path = os.path.join(WORK_DIR, "journey_cache.json")

# Check for pre-computed results
precomputed_path = os.path.join(TRAVEL_OUTPUT, f"candidate_reduction_scores(n={len(candidate_stations)}).csv")
precomputed_50 = os.path.join(TRAVEL_OUTPUT, "candidate_reduction_scores(n=50).csv")

USE_PRECOMPUTED = False
if os.path.exists(precomputed_path):
    print(f"Found pre-computed results: {precomputed_path}")
    USE_PRECOMPUTED = True
elif os.path.exists(precomputed_50):
    print(f"Found pre-computed results (n=50): {precomputed_50}")
    print("Will use n=50 results — re-run with API access to get full 108.")
    USE_PRECOMPUTED = True
    precomputed_path = precomputed_50


In [ ]:
# ─── 4.2 Helper functions ───

def strip_rail_suffix(name):
    return re.sub(r'\s+(NR|LU)$', '', str(name).strip(), flags=re.IGNORECASE)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

print("Travel time helper functions defined.")


In [ ]:
# ─── 4.3 Build network and compute travel times ───

if USE_PRECOMPUTED:
    print(f"Loading pre-computed travel time results from {precomputed_path}")
    travel_results = pd.read_csv(precomputed_path)
    print(f"Loaded {len(travel_results)} results")
    print(f"Non-zero reduction: {(travel_results['total_reduction'] > 0).sum()}")
else:
    print("Computing travel times from TfL API (this takes ~1 hour)...")
    
    # Load link frequency data
    lf = pd.read_csv(link_freq_path)
    lf.columns = lf.columns.str.strip()
    lf["From Station Clean"] = lf["From Station"].apply(strip_rail_suffix)
    lf["To Station Clean"] = lf["To Station"].apply(strip_rail_suffix)
    lf["Total"] = (lf["Total"].astype(str).str.replace(",", "")
                    .str.strip().pipe(pd.to_numeric, errors="coerce")
                    .fillna(0).astype(int))

    from_totals = lf.groupby("From Station Clean")["Total"].sum()
    to_totals = lf.groupby("To Station Clean")["Total"].sum()
    station_population = (from_totals.add(to_totals, fill_value=0)
                          .reset_index()
                          .rename(columns={"From Station Clean": "Station", 0: "Population"}))
    station_population.columns = ["Station", "Population"]
    
    print(f"Unique stations: {len(station_population)}")

    # Fetch station coordinates from TfL API
    def get_station_coords(station_name):
        import requests
        url = f"https://api.tfl.gov.uk/StopPoint/Search/{requests.utils.quote(station_name)}?app_key={tfl_api_key}"
        try:
            r = requests.get(url, timeout=10).json()
            matches = r.get("matches", [])
            if matches:
                return (matches[0]["lat"], matches[0]["lon"])
        except Exception:
            pass
        return None

    all_stations_t = set(lf["From Station Clean"].tolist() + lf["To Station Clean"].tolist())
    station_coords_t = {}
    for station in all_stations_t:
        coords = get_station_coords(station)
        if coords:
            station_coords_t[station] = coords
        time.sleep(0.05)
    print(f"Fetched {len(station_coords_t)} / {len(all_stations_t)} station coords")

    # Journey cache
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            journey_cache = json.load(f)
        print(f"Loaded {len(journey_cache)} cached journey times")
    else:
        journey_cache = {}

    def get_journey_time(from_c, to_c):
        import requests
        key = f"{from_c[0]},{from_c[1]}|{to_c[0]},{to_c[1]}"
        if key in journey_cache:
            return journey_cache[key]
        url = (f"https://api.tfl.gov.uk/Journey/JourneyResults/"
               f"{from_c[0]},{from_c[1]}/to/{to_c[0]},{to_c[1]}?app_key={tfl_api_key}")
        try:
            r = requests.get(url, timeout=10).json()
            secs = r["journeys"][0]["duration"] * 60
        except Exception:
            secs = None
        journey_cache[key] = secs
        time.sleep(0.05)
        return secs

    # Fetch edge times
    link_pairs = lf[["From Station Clean", "To Station Clean"]].drop_duplicates().values.tolist()
    edge_times = {}
    for frm, to in link_pairs:
        if frm in station_coords_t and to in station_coords_t:
            t = get_journey_time(station_coords_t[frm], station_coords_t[to])
            if t:
                edge_times[(frm, to)] = t

    with open(cache_path, "w") as f:
        json.dump(journey_cache, f)

    # Build graph
    G = nx.DiGraph()
    for station, (lat, lon) in station_coords_t.items():
        G.add_node(station, lat=lat, lon=lon)
    for (frm, to), t in edge_times.items():
        G.add_edge(frm, to, travel_time=t)

    # Stitch disconnected components
    components = list(nx.weakly_connected_components(G))
    largest_cc = max(components, key=len)
    for component in components:
        if component == largest_cc:
            continue
        for station in component:
            if station not in station_coords_t:
                continue
            slat, slon = station_coords_t[station]
            nearest = min(
                [(s, haversine_m(slat, slon, station_coords_t[s][0], station_coords_t[s][1]))
                 for s in largest_cc if s in station_coords_t],
                key=lambda x: x[1]
            )
            nb, dist = nearest
            t = get_journey_time((slat, slon), (station_coords_t[nb][0], station_coords_t[nb][1]))
            t = t or dist / 11.1
            G.add_edge(station, nb, travel_time=t)
            G.add_edge(nb, station, travel_time=t)

    print(f"Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

    # Evaluate candidates
    def get_nearby(G_base, lat, lon, radius_km, min_st=min_nearby_stations):
        r = radius_km
        while True:
            nearby = [s for s, d in G_base.nodes(data=True)
                      if haversine_m(lat, lon, d["lat"], d["lon"]) / 1000 <= r]
            if len(nearby) >= min_st or r > 20:
                return nearby
            r += 0.5

    def add_candidate(G_base, lat, lon, label, n_conn):
        G_new = copy.deepcopy(G_base)
        G_new.add_node(label, lat=lat, lon=lon)
        distances = sorted(
            [(s, haversine_m(lat, lon, d["lat"], d["lon"]))
             for s, d in G_new.nodes(data=True) if s != label],
            key=lambda x: x[1]
        )
        for nb, _ in distances[:n_conn]:
            nb_lat, nb_lon = G_new.nodes[nb]["lat"], G_new.nodes[nb]["lon"]
            t_to = get_journey_time((lat, lon), (nb_lat, nb_lon))
            t_from = get_journey_time((nb_lat, nb_lon), (lat, lon))
            with open(cache_path, "w") as f:
                json.dump(journey_cache, f)
            t_to = t_to or haversine_m(lat, lon, nb_lat, nb_lon) / 11.1
            t_from = t_from or haversine_m(nb_lat, nb_lon, lat, lon) / 11.1
            G_new.add_edge(label, nb, travel_time=t_to)
            G_new.add_edge(nb, label, travel_time=t_from)
        return G_new

    def compute_reduction(G_base, G_new, sp, c_lat, c_lon, r_km):
        pop_dict = dict(zip(sp["Station"], sp["Population"]))
        nearby = get_nearby(G_base, c_lat, c_lon, r_km)
        origins = [s for s in nearby if s in pop_dict and s in G_base.nodes]
        total_red = 0.0
        for origin in origins:
            pop = pop_dict[origin]
            d_old = nx.single_source_dijkstra_path_length(G_base, source=origin, weight="travel_time")
            d_new = nx.single_source_dijkstra_path_length(G_new, source=origin, weight="travel_time")
            for dest in d_old:
                if dest == origin:
                    continue
                red = d_old[dest] - d_new.get(dest, d_old[dest])
                if red > 0:
                    total_red += pop * red
        return total_red

    results = []
    for lat, lon, label in candidate_stations:
        print(f"Evaluating: {label} ({lat:.4f}, {lon:.4f})")
        G_new = add_candidate(G, lat, lon, label, n_connections)
        total_red = compute_reduction(G, G_new, station_population, lat, lon, benefit_radius_km)
        print(f"  total reduction: {total_red:,.0f} person-seconds")
        results.append({"label": label, "lat": lat, "lon": lon, "total_reduction": total_red})

    travel_results = pd.DataFrame(results).sort_values("total_reduction", ascending=False)
    save_path = os.path.join(TRAVEL_OUTPUT, f"candidate_reduction_scores(n={len(travel_results)}).csv")
    travel_results.to_csv(save_path, index=False)
    print(f"\nSaved: {save_path}")

print(f"\nTravel time results: {len(travel_results)} candidates")
print(f"Non-zero reductions: {(travel_results['total_reduction'] > 0).sum()}")
travel_results.head(10)


---
## 5. Combined Multi-Criteria Scoring & Sensitivity Analysis

Merges all four component scores into a single suitability ranking.

**Weights (literature-based baseline):**
- Population demand (gravity): **0.35**
- Accessibility gap (connectivity): **0.30**
- Congestion relief (crowding): **0.20**
- Travel time reduction: **0.15**

References: Çalışkan (2023), Akin & Kara (2020), Litman (2024)


In [ ]:
# ─── 5.1 Normalisation helper ───
def minmax(series):
    s_min, s_max = series.min(), series.max()
    if s_max - s_min < 1e-12:
        return pd.Series(0.0, index=series.index)
    return (series - s_min) / (s_max - s_min)

# Weights
WEIGHTS = {
    "gravity": 0.35,
    "connectivity": 0.30,
    "crowding": 0.20,
    "travel_time": 0.15,
}
print(f"Weights: {WEIGHTS}")


In [ ]:
# ─── 5.2 Build combined grid ───
combined = grid_latlon[["lat", "lon", "gravity_log"]].copy().reset_index(drop=True)
combined["lat_r"] = combined["lat"].round(6)
combined["lon_r"] = combined["lon"].round(6)

# Gravity (already have it)
combined["gravity_norm"] = minmax(combined["gravity_log"])

# Crowding
crowd_df = candidates_crowd[["lat", "lon", "crowding_score", "crowding_score_norm",
                              "min_distance_to_station_km", "top_relieved_stations"]].copy()
crowd_df["lat_r"] = crowd_df["lat"].round(6)
crowd_df["lon_r"] = crowd_df["lon"].round(6)
combined = combined.merge(
    crowd_df[["lat_r", "lon_r", "crowding_score", "crowding_score_norm",
              "min_distance_to_station_km", "top_relieved_stations"]],
    on=["lat_r", "lon_r"], how="left"
)
combined["crowding_norm"] = minmax(combined["crowding_score"].fillna(0))

# Travel time
travel_results["lat_r"] = travel_results["lat"].round(6)
travel_results["lon_r"] = travel_results["lon"].round(6)
combined = combined.merge(
    travel_results[["lat_r", "lon_r", "total_reduction"]],
    on=["lat_r", "lon_r"], how="left"
)
combined["total_reduction"] = combined["total_reduction"].fillna(0)
combined["travel_time_norm"] = minmax(combined["total_reduction"])

# Connectivity
conn_df = conn_result.copy()
conn_df["lat_r4"] = conn_df["lat"].round(4)
conn_df["lon_r4"] = conn_df["lon"].round(4)
combined["lat_r4"] = combined["lat"].round(4)
combined["lon_r4"] = combined["lon"].round(4)
combined = combined.merge(
    conn_df[["lat_r4", "lon_r4", "connectivity_score",
             "harmonic_mean_adj_dist_norm", "line_unique_count_norm"]],
    on=["lat_r4", "lon_r4"], how="left"
)
combined.drop(columns=["lat_r4", "lon_r4"], inplace=True, errors="ignore")
combined["connectivity_norm"] = minmax(combined["connectivity_score"].fillna(0))
has_connectivity = combined["connectivity_score"].notna().sum() > 100

print(f"Grid points: {len(combined)}")
print(f"Gravity coverage:      {(combined['gravity_norm'] > 0).sum()} / {len(combined)}")
print(f"Connectivity coverage: {(combined['connectivity_norm'] > 0).sum()} / {len(combined)}")
print(f"Crowding coverage:     {(combined['crowding_norm'] > 0).sum()} / {len(combined)}")
print(f"Travel time coverage:  {(combined['travel_time_norm'] > 0).sum()} / {len(combined)}")


In [ ]:
# ─── 5.3 Combined score ───
MIN_STATION_DIST_KM = 0.8
if "min_distance_to_station_km" in combined.columns:
    too_close = combined["min_distance_to_station_km"] < MIN_STATION_DIST_KM
    print(f"Excluding {too_close.sum()} grid points within {MIN_STATION_DIST_KM}km of existing station")
else:
    too_close = pd.Series(False, index=combined.index)

combined["combined_score"] = (
    WEIGHTS["gravity"]      * combined["gravity_norm"]
    + WEIGHTS["connectivity"] * combined["connectivity_norm"]
    + WEIGHTS["crowding"]     * combined["crowding_norm"]
    + WEIGHTS["travel_time"]  * combined["travel_time_norm"]
)
combined.loc[too_close, "combined_score"] = 0.0
combined["rank"] = combined["combined_score"].rank(ascending=False, method="min").astype(int)
combined = combined.sort_values("rank")

# Print top 20
print("\nTOP 20 CANDIDATE LOCATIONS")
print("=" * 80)
for _, r in combined.head(20).iterrows():
    print(f"  #{int(r['rank']):>3d}  ({r['lat']:.4f}, {r['lon']:.4f})  combined={r['combined_score']:.4f}")
    print(f"        gravity={r['gravity_norm']:.3f}  connectivity={r['connectivity_norm']:.3f}  "
          f"crowding={r['crowding_norm']:.3f}  travel_time={r['travel_time_norm']:.3f}")
    if pd.notna(r.get("top_relieved_stations")) and r.get("top_relieved_stations"):
        print(f"        relieves: {r['top_relieved_stations']}")

# Save
output_cols = ["lat", "lon", "rank", "combined_score",
               "gravity_norm", "connectivity_norm", "crowding_norm", "travel_time_norm",
               "gravity_log", "crowding_score", "total_reduction",
               "min_distance_to_station_km", "top_relieved_stations"]
output_cols = [c for c in output_cols if c in combined.columns]

combined[output_cols].to_csv(os.path.join(FINAL_OUTPUT, "combined_scores_full.csv"), index=False)
combined.head(20)[output_cols].to_csv(os.path.join(FINAL_OUTPUT, "top20_combined.csv"), index=False)
print(f"\nSaved to {FINAL_OUTPUT}")


### 5.4 Sensitivity Analysis

In [ ]:
# ─── 5.4 Sensitivity analysis across weight scenarios ───
from collections import Counter

weight_scenarios = {
    "Literature baseline (0.35/0.30/0.20/0.15)": {
        "gravity": 0.35, "connectivity": 0.30, "crowding": 0.20, "travel_time": 0.15},
    "Equal (0.25 each)": {
        "gravity": 0.25, "connectivity": 0.25, "crowding": 0.25, "travel_time": 0.25},
    "Demand-dominant (0.55)": {
        "gravity": 0.55, "connectivity": 0.15, "crowding": 0.15, "travel_time": 0.15},
    "Connectivity-dominant (0.55)": {
        "gravity": 0.15, "connectivity": 0.55, "crowding": 0.15, "travel_time": 0.15},
    "Crowding-dominant (0.55)": {
        "gravity": 0.15, "connectivity": 0.15, "crowding": 0.55, "travel_time": 0.15},
    "Travel-time-dominant (0.55)": {
        "gravity": 0.15, "connectivity": 0.15, "crowding": 0.15, "travel_time": 0.55},
    "Demand + Connectivity (0.35/0.35/0.15/0.15)": {
        "gravity": 0.35, "connectivity": 0.35, "crowding": 0.15, "travel_time": 0.15},
    "Demand + Crowding (0.35/0.15/0.35/0.15)": {
        "gravity": 0.35, "connectivity": 0.15, "crowding": 0.35, "travel_time": 0.15},
    "Connectivity + Crowding (0.15/0.35/0.35/0.15)": {
        "gravity": 0.15, "connectivity": 0.35, "crowding": 0.35, "travel_time": 0.15},
}

sensitivity_results = []
top5_counter = Counter()

for label, w in weight_scenarios.items():
    score = (
        w["gravity"]      * combined["gravity_norm"]
        + w["connectivity"] * combined["connectivity_norm"]
        + w["crowding"]     * combined["crowding_norm"]
        + w["travel_time"]  * combined["travel_time_norm"]
    )
    score.loc[too_close] = 0.0
    top5_idx = score.nlargest(5).index
    top5 = combined.loc[top5_idx, ["lat", "lon"]].copy()
    top5["score"] = score.loc[top5_idx].values
    top5["scenario"] = label
    sensitivity_results.append(top5)

    for _, r in top5.iterrows():
        top5_counter[f"({r['lat']:.4f}, {r['lon']:.4f})"] += 1

    print(f"\n{label}:")
    for i, (_, r) in enumerate(top5.iterrows(), 1):
        print(f"  {i}. ({r['lat']:.4f}, {r['lon']:.4f})  score={r['score']:.4f}")

sensitivity_df = pd.concat(sensitivity_results, ignore_index=True)
sensitivity_df.to_csv(os.path.join(FINAL_OUTPUT, "sensitivity_analysis.csv"), index=False)

# Robustness summary
n_scenarios = len(weight_scenarios)
print(f"\n\nROBUSTNESS SUMMARY (top-5 appearances across {n_scenarios} scenarios)")
print("=" * 60)
for loc, count in top5_counter.most_common(15):
    print(f"  {loc}  appears {count}/{n_scenarios} ({count/n_scenarios*100:.0f}%)")


### 5.5 Combined Score Heatmap

In [ ]:
# ─── Combined score city-wide map ───
m_combined = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_combined)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_combined)

vmin_comb = float(combined["combined_score"].min())
vmax_comb = float(combined["combined_score"].max())
cmap_combined = cm.LinearColormap(
    ["#08306b", "#2171b5", "#6baed6", "#ef6548", "#cb181d", "#99000d"],
    vmin=vmin_comb, vmax=vmax_comb,
)
cmap_combined.caption = "Combined Score"

for _, r in combined.iterrows():
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=5.0,
        color=cmap_combined(float(r["combined_score"])),
        fill=True, fill_opacity=0.7, weight=0,
        popup=(
            f"rank={int(r['rank'])}<br>"
            f"combined={r['combined_score']:.4f}<br>"
            f"gravity={r['gravity_norm']:.3f}<br>"
            f"connectivity={r['connectivity_norm']:.3f}<br>"
            f"crowding={r['crowding_norm']:.3f}<br>"
            f"travel_time={r['travel_time_norm']:.3f}"
        ),
    ).add_to(m_combined)

cmap_combined.add_to(m_combined)
folium.LayerControl(collapsed=False).add_to(m_combined)

print("Combined score city-wide heatmap:")
m_combined


### 5.6 Final Top 10 Map

In [ ]:
# ─── Final top 10 map ───
top10_final = combined.head(10).reset_index(drop=True)

m_final = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles=None)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_final)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_final)

# Background heatmap
for _, r in combined.iterrows():
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=4.0,
        color=cmap_combined(float(r["combined_score"])),
        fill=True, fill_opacity=0.2, weight=0,
    ).add_to(m_final)

# Top 10
for rank, (_, row) in enumerate(top10_final.iterrows(), start=1):
    lat, lon = float(row["lat"]), float(row["lon"])
    relieves = row.get("top_relieved_stations", "")
    if pd.isna(relieves):
        relieves = ""

    folium.CircleMarker(
        location=[lat, lon], radius=18,
        color="#d7191c", fill=True, fill_color="#d7191c",
        fill_opacity=0.85, weight=2,
        popup=folium.Popup(
            f"<b>Rank #{rank}</b><br>"
            f"combined={row['combined_score']:.4f}<br>"
            f"gravity={row['gravity_norm']:.3f}<br>"
            f"connectivity={row['connectivity_norm']:.3f}<br>"
            f"crowding={row['crowding_norm']:.3f}<br>"
            f"travel_time={row['travel_time_norm']:.3f}<br>"
            f"{relieves}",
            max_width=300,
        ),
    ).add_to(m_final)
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:13px;font-weight:bold;color:white;text-align:center;line-height:30px;width:30px;height:30px;">{rank}</div>',
            icon_size=(30, 30), icon_anchor=(15, 15),
        ),
    ).add_to(m_final)

folium.LayerControl(collapsed=False).add_to(m_final)
print("Final top 10 combined score map:")
m_final


---
## Summary

This notebook runs the full pipeline:
1. **Gravity Model** → identifies high-demand areas
2. **Connectivity Model** → identifies poorly-served areas
3. **Crowding Model** → identifies congestion relief opportunities
4. **Travel Time Model** → estimates journey-time improvements
5. **Combined Scoring** → ranks all candidates with sensitivity analysis

Check the `final_weighting_score/output/` folder for all CSV outputs.
